# Pillar 2: Closures, Function Wrappers & Decorators

## Core Mechanics & Theory

Decorators are everywhere in high-performance Python, AI, and kernel libraries (e.g., `@triton.jit`, `@torch.compile`, `@functools.lru_cache`). 

**Under the hood**, decorators are just functions that take a function as input and return a modified function.

### 1. Closures: How Functions Remember State

When an inner function references a variable from an outer function, that variable is called a **free variable**. Python packages the inner function and the free variable together into a **closure**.

**Key Details:**

- The variable is stored in the function's `__closure__` attribute as a **cell object**
- The inner function **remembers this state** even after the outer function finishes executing and returns

In [57]:
def make_multiplier(factor):
    # 'factor' is in the enclosing scope
    def multiply(x):
        return x * factor  # 'factor' is captured in the closure
    return multiply

double = make_multiplier(2)
print(double(5))  # Output: 10
print(double.__closure__[0].cell_contents)  # Output: 2

10
2


### 2. The Mechanics of a Basic Decorator

A **decorator intercepts** a function call to execute code before and after the target function runs.

**The @decorator Syntax**

The `@decorator` syntax is pure syntactic sugar:

```python
@my_decorator
def my_func():
    pass
```

is equivalent to:

```python
my_func = my_decorator(my_func)
```

In [58]:
def my_decorator(func):
    def wrapper(*args,**kwargs):
        print("before calling func")
        
        func(*args,**kwargs)

        print("after calling func")
    return wrapper

In [59]:
@my_decorator
def calculate(x):
    print(x*2)

# Is 100% equivalent to:
# calculate = my_decorator(calculate)

calculate(2)

before calling func
4
after calling func


In [60]:
import time

def timer(func):
    
    def wrapper(*args,**kwargs):
        start = time.time()
        
        func(*args,**kwargs)
        
        end = time.time()
        
        print(f"time taken: {end-start}")
    return wrapper

In [61]:
@timer
def clock(stop = 10):
    time.sleep(stop)
    print(f"done time {stop}")

clock(2)

done time 2
time taken: 2.000702142715454


### 3. Preserving Function Identity (@functools.wraps)

**The Problem**

When you wrap a function, its `__name__` and `__doc__` get replaced by the wrapper's name and docstring. This causes issues in:
- Debugging
- Profiling
- Frameworks

**The Solution**

`functools.wraps` copies the original name, module, and docstring back onto the wrapper, preserving the original function's identity.

In [62]:
import functools
import time

def timer(func):
    @functools.wraps(func)  # Keeps original __name__ and docstring intact
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"[{func.__name__}] executed in {elapsed:.6f}s")
        return result
    return wrapper

@timer
def heavy_compute(n):
    """Computes sum of squares."""
    return sum(i * i for i in range(n))

print(heavy_compute(100_000))
print(heavy_compute.__name__)  # Prints 'heavy_compute', NOT 'wrapper'

[heavy_compute] executed in 0.009901s
333328333350000
heavy_compute


### 4. Parameterized Decorators (Decorators with Arguments)

Sometimes you need to configure a decorator (e.g., `@retry(times=3)` or `@triton.jit(launch_bounds=...)`).

To accept arguments, you need **3 levels of functions**:

1. **Outer Factory**: Takes the configuration arguments (e.g., `times`)
2. **Middle Decorator**: Takes the target function (`func`)
3. **Inner Wrapper**: Takes the function arguments (`*args`, `**kwargs`) and executes the call

This nesting allows you to parameterize the decorator's behavior!

In [63]:
def repeat(num_times):
    def decorator_repeat(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(num_times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator_repeat

@repeat(num_times=3)
def greet(name):
    print(f"Hello, {name}!")

greet("Alex")  # Prints "Hello, Alex!" 3 times

Hello, Alex!
Hello, Alex!
Hello, Alex!


## Exercise 1: Custom In-Memory Memoization / Cache Decorator

Write a caching decorator `memoize(func)` from scratch **without importing** `functools.lru_cache`:

### Requirements:

- Maintain a dictionary `cache = {}` inside the closure
- The wrapper must construct a cache key from incoming arguments
  - Example: `key = (args, tuple(sorted(kwargs.items())))`
- **If key is in cache**: print `"[Cache Hit] <key>"` and return the cached result
- **If not**: execute the original function, store the result in cache, print `"[Cache Miss] <key>"`, and return the result
- Use `@functools.wraps` to preserve function metadata

In [64]:
def local_cache(func):
    cache={}
    def wrapper(*args,**kwargs):
        
        key = (args,tuple(sorted(*kwargs.items())))
        
        if key in cache:
            print("[CACHE HIT]")
            return cache[key]
        val = func(*args,**kwargs)
        print(val)
        
        cache[key]=val
    return wrapper

In [65]:
@local_cache
def name_print(num, name="tyler"):
    return " ".join([name for i in range(num)])

In [66]:
name_print(3,name="haha")

haha haha haha


In [67]:
name_print(3,name="haha")

[CACHE HIT]


'haha haha haha'

## Exercise 2: Parameterized Kernel / Function Call Counter

Write a parameterized decorator `call_counter(limit: int)`:

### Requirements:

- Keep track of how many times the wrapped function has been called
- **If total calls exceed limit**: raise `RuntimeError(f"Call limit of {limit} exceeded for {func.__name__}")`
- **If within limit**: 
  - Increment the count
  - Print `"[Call {count}/{limit}] {func.__name__}"`
  - Execute the function

In [68]:
import functools
def call_counter(limit: int):
    def decorator(func):
        counter = 0
        @functools.wraps(func)
        def wrapper(*args,**kwargs):
            nonlocal counter
            if counter>=limit:
                raise RuntimeError(f"call limit reached for {func.__name__}")
            counter+=1
            return func(*args,**kwargs)
            
        return wrapper
    return decorator
            
        

In [69]:
@call_counter(3)
def printer(num):
    print(f"calling {i}")
    
for i in range(4):
    printer(i)

calling 0
calling 1
calling 2


RuntimeError: call limit reached for printer